In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


# Simple main-sequence stellar structure solver
#
# The standard system of 5 equations:
#   d(rho)/dr, dT/dr, dM/dr, dL/dr, d(tau)/dr
#
# shooting method:
#   - choose central temperature Tc
#   - vary central density rho_c until the surface luminosity
#     condition L = 4*pi*sigma*R^2*T^4 is satisfied


LOW_MASS_FILENAME = "lowmass_star.txt"
HIGH_MASS_FILENAME = "highmass_star.txt"
DATA_FOLDER = os.getcwd()  # Folder containing the example .txt files

# Physical constants
G = 6.67430e-11
k_B = 1.380649e-23
hbar = 1.054571817e-34
m_e = 9.10938356e-31
m_p = 1.67262192369e-27
c = 2.99792458e8
sigma_sb = 5.670374419e-8
pi = np.pi
a_rad = 4.0 * sigma_sb / c

# Solar values for plotting
M_sun = 1.98847e30
R_sun = 6.957e8
L_sun = 3.828e26

# Composition
X = 0.7381
Y = 0.2485
Z = 0.0134
X_CNO = 0.03 * X

# Mean molecular weight for fully ionized gas
mu = 1.0 / (2.0 * X + 0.75 * Y + 0.5 * Z)

# Adiabatic index used in convective gradient
gamma_ad = 5.0 / 3.0


# DM Heating setup:
# Keep the standard solver as the baseline and add dark matter only as an
# extra source term in the luminosity equation:
#   dL/dr = 4*pi*r^2*rho*epsilon_nuc + 4*pi*r^2*P_DM(r)
# where P_DM is the annihilation power density.

pc = 3.085677581491367e16
GeV_to_kg = 1.78266192e-27
sigma_v_DM_default = 3.0e-32
section57_s_default = 1.25

RUN_SECTION_57_COMPARISON = True



# Helpful functions

def pressure_total(rho, T):
    """Total pressure = degeneracy + ideal gas + radiation."""
    # Non-relativistic electron degeneracy pressure
    P_deg = ((3.0 * pi**2)**(2.0 / 3.0) / 5.0) * (hbar**2 / m_e) * (rho / m_p)**(5.0 / 3.0)

    # Ideal gas pressure
    P_gas = rho * k_B * T / (mu * m_p)

    # Radiation pressure
    P_rad = (1.0 / 3.0) * a_rad * T**4

    return P_deg + P_gas + P_rad


def pressure_parts(rho, T):
    """Return pressure pieces separately for plotting."""
    P_deg = ((3.0 * pi**2)**(2.0 / 3.0) / 5.0) * (hbar**2 / m_e) * (rho / m_p)**(5.0 / 3.0)
    P_gas = rho * k_B * T / (mu * m_p)
    P_rad = (1.0 / 3.0) * a_rad * T**4
    return P_deg, P_gas, P_rad


def dP_drho(rho, T):
    """Partial derivative dP/drho at fixed T."""
    term_deg = ((3.0 * pi**2)**(2.0 / 3.0) / 3.0) * (hbar**2 / (m_e * m_p)) * (rho / m_p)**(2.0 / 3.0)
    term_gas = k_B * T / (mu * m_p)
    return term_deg + term_gas


def dP_dT(rho, T):
    """Partial derivative dP/dT at fixed rho."""
    term_gas = rho * k_B / (mu * m_p)
    term_rad = (4.0 / 3.0) * a_rad * T**3
    return term_gas + term_rad


def epsilon_pp(rho, T):
    """PP-chain energy generation in W/kg."""
    rho5 = rho / 1.0e5
    T6 = T / 1.0e6
    if T6 <= 0:
        return 0.0
    return 1.07e-7 * rho5 * X**2 * T6**4


def epsilon_cno(rho, T):
    """CNO-cycle energy generation in W/kg."""
    rho5 = rho / 1.0e5
    T6 = T / 1.0e6
    if T6 <= 0:
        return 0.0
    return 8.24e-26 * rho5 * X * X_CNO * T6**19.9



def epsilon_total(rho, T):
    """Standard nuclear specific energy generation (W/kg)."""
    return epsilon_pp(rho, T) + epsilon_cno(rho, T)


def make_dm_case(rho_dm0, m_dm_GeV, s=section57_s_default,
                 sigma_v_DM=sigma_v_DM_default, label=None):
    """Create one dark-matter case."""
    if label is None:
        if m_dm_GeV >= 1.0e3:
            mass_text = f"{m_dm_GeV/1.0e3:g} TeV"
        else:
            mass_text = f"{m_dm_GeV:g} GeV"
        label = f"DM: rho0={rho_dm0:.1e} kg/m^3, m={mass_text}, s={s:g}"

    return {
        'enabled': True,
        'rho_dm0': float(rho_dm0),
        'm_dm_GeV': float(m_dm_GeV),
        'm_dm': float(m_dm_GeV) * GeV_to_kg,
        's': float(s),
        'sigma_v_DM': float(sigma_v_DM),
        'label': label,
    }


def rho_DM_profile(r, dm_case=None):
    """Dark-matter cusp density rho_DM(r) in kg/m^3."""
    if not dm_case or not dm_case.get('enabled', False):
        return 0.0
    r_safe = max(float(r), 1.0)
    return dm_case['rho_dm0'] * (r_safe / pc) ** (-dm_case['s'])


def power_density_DM(r, dm_case=None):
    """Dark-matter annihilation power density P_DM(r) in W/m^3."""
    if not dm_case or not dm_case.get('enabled', False):
        return 0.0
    rho_dm = rho_DM_profile(r, dm_case)
    m_dm = max(dm_case['m_dm'], 1.0e-99)
    return dm_case['sigma_v_DM'] * rho_dm**2 * c**2 / m_dm


def dLdr_DM(r, dm_case=None):
    """Extra luminosity source from DM heating, in W/m.

    We are given a volumetric power density P_DM, so the most direct
    practical implementation is to add 4*pi*r^2*P_DM to dL/dr.
    """
    if not dm_case or not dm_case.get('enabled', False):
        return 0.0
    r_safe = max(float(r), 1.0)
    return 4.0 * pi * r_safe**2 * power_density_DM(r_safe, dm_case)


def initial_DM_luminosity(r0, dm_case=None):
    """Analytic DM luminosity enclosed within the starting radius r0."""
    if not dm_case or not dm_case.get('enabled', False):
        return 0.0

    s = dm_case['s']
    if abs(3.0 - 2.0 * s) < 1.0e-12:
        return dLdr_DM(max(r0, 1.0), dm_case) * max(r0, 1.0)

    A = (
        dm_case['sigma_v_DM']
        * c**2
        * dm_case['rho_dm0']**2
        * pc**(2.0 * s)
        / max(dm_case['m_dm'], 1.0e-99)
    )
    return 4.0 * pi * A * max(r0, 1.0)**(3.0 - 2.0 * s) / (3.0 - 2.0 * s)


def representative_section57_cases():
    """Five practical comparison cases spanning weak, mixed, and strong DM heating."""
    return [
        # low density + low DM mass
        make_dm_case(1.0e-20, 100.0, s=1.25),
        # intermediate density + low DM mass
        make_dm_case(1.0e-15, 100.0, s=1.25),
        # high density + high DM mass
        make_dm_case(1.0e-11, 1.0e5, s=1.25),
        # added: low density + high DM mass
        make_dm_case(1.0e-20, 1.0e5, s=1.25),
        # added: high density + low DM mass
        make_dm_case(1.0e-11, 100.0, s=1.25),
    ]


def opacity_parts(rho, T):
    """Return kappa_es, kappa_ff, kappa_Hminus separately."""
    rho3 = max(rho / 1.0e3, 1.0e-30)
    T = max(T, 1.0)

    k_es = 0.02 * (1.0 + X)
    k_ff = 1.0e24 * (Z + 0.0001) * rho3**0.7 * T**(-3.5)
    k_hm = 2.5e-32 * (Z / 0.02) * rho3**0.5 * T**9
    return k_es, k_ff, k_hm


def opacity_total(rho, T):
    """Combined opacity formula."""
    k_es, k_ff, k_hm = opacity_parts(rho, T)
    k_highT = max(k_es, k_ff)
    # Smooth combination that switches to H- near the surface
    return 1.0 / (1.0 / max(k_hm, 1.0e-40) + 1.0 / max(k_highT, 1.0e-40))



def effective_temperature_from_surface(surface):
    """Compute Teff from luminosity and radius using Stefan-Boltzmann."""
    radius = max(surface['r'], 1.0e-30)
    luminosity = max(surface['L'], 0.0)
    return (luminosity / (4.0 * pi * sigma_sb * radius**2))**0.25


def get_surface_teff(surface):
    """Return Teff if stored; otherwise compute it from L and R."""
    if surface is None:
        return np.nan
    if 'Teff' in surface and np.isfinite(surface['Teff']):
        return surface['Teff']
    return effective_temperature_from_surface(surface)


def convective_from_dlogP_dlogT(dlogP_dlogT):
    """Approximate convective zones from the tabulated stability ratio."""
    threshold = gamma_ad / (gamma_ad - 1.0)
    return dlogP_dlogT <= threshold + 1.0e-8


def load_reference_star_file(file_path, label):
    """Load one example-star text file and convert cgs columns to SI units."""
    raw = np.loadtxt(file_path, comments='#')

    # Convert the columns used by the plots from cgs to SI.
    r = raw[:, 1] * 1.0e-2
    rho = raw[:, 2] * 1.0e3
    T = raw[:, 3]
    M_r = raw[:, 4] * 1.0e-3
    L_r = raw[:, 5] * 1.0e-7
    dLdr = raw[:, 6] * 1.0e-5
    dLppdr = raw[:, 7] * 1.0e-5
    dLcnodr = raw[:, 8] * 1.0e-5
    dlogP_dlogT = raw[:, 9]

    kappa = raw[:, 15] * 0.1
    k_hm = raw[:, 16] * 0.1
    k_ff = raw[:, 17] * 0.1
    k_es = raw[:, 18] * 0.1

    P = raw[:, 19] * 0.1
    P_deg = raw[:, 20] * 0.1
    P_gas = raw[:, 21] * 0.1
    P_rad = raw[:, 22] * 0.1

    shell_factor = 4.0 * pi * np.maximum(r, 1.0)**2 * np.maximum(rho, 1.0e-30)
    eps_pp = dLppdr / shell_factor
    eps_cno = dLcnodr / shell_factor

    model = {
        'label': label,
        'r': r,
        'rho': rho,
        'T': T,
        'M': M_r,
        'L': L_r,
        'P': P,
        'P_deg': P_deg,
        'P_gas': P_gas,
        'P_rad': P_rad,
        'kappa': kappa,
        'k_es': k_es,
        'k_ff': k_ff,
        'k_hm': k_hm,
        'dLdr': dLdr,
        'dLdr_pp': dLppdr,
        'dLdr_cno': dLcnodr,
        'dLdr_dm': np.zeros_like(dLdr),
        'eps_pp': eps_pp,
        'eps_cno': eps_cno,
        'rho_dm': np.zeros_like(r),
        'P_dm': np.zeros_like(r),
        'convective': convective_from_dlogP_dlogT(dlogP_dlogT),
        'surface': {
            'r': r[-1],
            'rho': rho[-1],
            'T': T[-1],
            'M': M_r[-1],
            'L': L_r[-1],
        },
    }
    model['surface']['Teff'] = effective_temperature_from_surface(model['surface'])
    model['dm_case'] = None
    return model


def load_example_stars_once(data_folder):
    """Load the low-mass and high-mass example files."""
    low_path = os.path.join(data_folder, LOW_MASS_FILENAME)
    high_path = os.path.join(data_folder, HIGH_MASS_FILENAME)

    if not os.path.exists(low_path):
        raise FileNotFoundError(f'Could not find {low_path}')
    if not os.path.exists(high_path):
        raise FileNotFoundError(f'Could not find {high_path}')

    example_stars = {
        'low': load_reference_star_file(low_path, 'Low-Mass Reference Star'),
        'high': load_reference_star_file(high_path, 'High-Mass Reference Star'),
    }
    return example_stars

# Note: The exclusion criteria later on will automatically disqualify the high mass star example.

# ODE system
# state = [rho, T, M, L, tau]

def stellar_derivatives(r, state, dm_case=None):
    rho, T, M, L, tau = state

    rho = max(rho, 1.0e-12)
    T = max(T, 1.0)
    M = max(M, 1.0e-30)
    r = max(r, 1.0)

    P = pressure_total(rho, T)
    kappa = opacity_total(rho, T)
    eps_nuc = epsilon_total(rho, T)

    dTdr_rad = -(3.0 * kappa * rho * L) / (16.0 * pi * a_rad * c * T**3 * r**2)
    dTdr_conv = -((1.0 - 1.0 / gamma_ad) * T / P) * (G * M * rho / r**2)

    if abs(dTdr_rad) < abs(dTdr_conv):
        dTdr = dTdr_rad
    else:
        dTdr = dTdr_conv

    dp_rho = dP_drho(rho, T)
    dp_T = dP_dT(rho, T)
    drhodr = -(G * M * rho / r**2 + dp_T * dTdr) / dp_rho

    dMdr = 4.0 * pi * r**2 * rho
    dLdr_nuc = 4.0 * pi * r**2 * rho * eps_nuc
    dLdr = dLdr_nuc + dLdr_DM(r, dm_case=dm_case)
    dtaudr = kappa * rho

    return np.array([drhodr, dTdr, dMdr, dLdr, dtaudr], dtype=float)

def rk4_step(r, y, h, dm_case=None):
    """One RK4 step."""
    k1 = stellar_derivatives(r, y, dm_case=dm_case)
    k2 = stellar_derivatives(r + 0.5 * h, y + 0.5 * h * k1, dm_case=dm_case)
    k3 = stellar_derivatives(r + 0.5 * h, y + 0.5 * h * k2, dm_case=dm_case)
    k4 = stellar_derivatives(r + h, y + h * k3, dm_case=dm_case)
    return y + (h / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def interpolate_surface_from_tau(history):
    """
    Estimate the stellar photosphere using the optical-depth condition.

    Method:
    - work with remaining optical depth to infinity
    - interpolate in log(remaining optical depth), which is more natural
      for the steep outer atmosphere
    - allow a small final extrapolation if the photosphere lies just
      beyond the last grid point
    """
    r = history['r']
    rho = history['rho']
    T = history['T']
    M = history['M']
    L = history['L']
    tau = history['tau']

    if len(r) < 2:
        return None

    target_remaining_tau = 2.0 / 3.0

    # Estimate remaining optical depth beyond the final grid point
    tau_end = tau[-1]
    delta_tau_remaining = 0.0
    H_rho_end = None

    dr_end = r[-1] - r[-2]
    if dr_end > 0:
        rho_end = max(rho[-1], 1.0e-30)
        T_end = max(T[-1], 1.0)
        drhodr_end = (rho[-1] - rho[-2]) / dr_end

        if drhodr_end < -1.0e-40:
            H_rho_end = rho_end / abs(drhodr_end)
            kappa_end = opacity_total(rho_end, T_end)
            delta_tau_remaining = kappa_end * rho_end * H_rho_end

            if not np.isfinite(delta_tau_remaining) or delta_tau_remaining < 0.0:
                delta_tau_remaining = 0.0
        else:
            H_rho_end = max(dr_end, 1.0)

    tau_infinity_est = tau_end + delta_tau_remaining

    # Remaining optical depth outside each radius point
    tau_remaining = tau_infinity_est - tau

    # Case 1: surface lies inside the computed grid
    if tau_remaining[0] > target_remaining_tau and tau_remaining[-1] <= target_remaining_tau:
        j = np.where(tau_remaining <= target_remaining_tau)[0][0]
        i = j - 1

        rem0 = max(tau_remaining[i], 1.0e-300)
        rem1 = max(tau_remaining[i + 1], 1.0e-300)

        # Log interpolation in remaining optical depth
        if rem0 != rem1:
            frac = (
                np.log(target_remaining_tau) - np.log(rem0)
            ) / (
                np.log(rem1) - np.log(rem0)
            )
        else:
            frac = 0.0

        frac = min(max(frac, 0.0), 1.0)

        def interp(arr):
            return arr[i] + frac * (arr[i + 1] - arr[i])

        surface = {
            'r': interp(r),
            'rho': interp(rho),
            'T': interp(T),
            'M': interp(M),
            'L': interp(L),
            'tau': tau_infinity_est - target_remaining_tau,
            'tau_end': tau_end,
            'tau_infinity_est': tau_infinity_est,
            'delta_tau_remaining': delta_tau_remaining,
            'tau_remaining_surface': target_remaining_tau,
            'index': i,
            'frac': frac,
        }
        return surface

    # Case 2: surface lies just beyond the last computed point
    if tau_remaining[-1] > target_remaining_tau and delta_tau_remaining > target_remaining_tau:
        rho_end = max(rho[-1], 1.0e-30)
        T_end = max(T[-1], 1.0)

        if H_rho_end is None or H_rho_end <= 0.0:
            H_rho_end = max(dr_end, 1.0)

        delta_r = H_rho_end * np.log(delta_tau_remaining / target_remaining_tau)

        # Keep extrapolation modest and stable
        delta_r = min(max(delta_r, 0.0), 3.0 * H_rho_end)

        r_surface = r[-1] + delta_r
        rho_surface = rho_end * np.exp(-delta_r / H_rho_end)

        surface = {
            'r': r_surface,
            'rho': rho_surface,
            'T': T[-1],
            'M': M[-1],
            'L': L[-1],
            'tau': tau_infinity_est - target_remaining_tau,
            'tau_end': tau_end,
            'tau_infinity_est': tau_infinity_est,
            'delta_tau_remaining': delta_tau_remaining,
            'tau_remaining_surface': target_remaining_tau,
            'index': len(r) - 2,
            'frac': 1.0,
        }
        return surface

    # Fallback: use the last point if something unusual happens
    return {
        'r': r[-1],
        'rho': rho[-1],
        'T': T[-1],
        'M': M[-1],
        'L': L[-1],
        'tau': tau[-1],
        'tau_end': tau_end,
        'tau_infinity_est': tau_infinity_est,
        'delta_tau_remaining': delta_tau_remaining,
        'tau_remaining_surface': tau_remaining[-1],
        'index': len(r) - 2,
        'frac': 1.0,
    }


def choose_step_size(r, T):
    """
    Small steps near the surface so the photosphere is resolved better.
    """
    if T > 1.0e6:
        frac = 0.01      # deep interior
    elif T > 1.0e5:
        frac = 0.003     # intermediate layers
    else:
        frac = 0.0005    # outer envelope / near photosphere

    return max(1.0e3, frac * r)


# Solve one trial star for fixed rho_c and T_c

def integrate_star(rho_c, T_c, max_steps=200000, dm_case=None):
    """
    Integrate outward from a small starting radius r0.

    - add DM heating only as an extra source in the luminosity equation
    - initialize L(r0) with nuclear + DM power inside r0
    """
    r0 = 1.0e3

    eps_c = epsilon_total(rho_c, T_c)
    M0 = (4.0 / 3.0) * pi * r0**3 * rho_c
    L0_nuc = (4.0 / 3.0) * pi * r0**3 * rho_c * eps_c
    L0_dm = initial_DM_luminosity(r0, dm_case=dm_case)
    L0 = L0_nuc + L0_dm
    y = np.array([rho_c, T_c, M0, L0, 0.0], dtype=float)

    r_list = [r0]
    rho_list = [y[0]]
    T_list = [y[1]]
    M_list = [y[2]]
    L_list = [y[3]]
    tau_list = [y[4]]

    converged_outer = False
    r = r0

    for _ in range(max_steps):
        h = choose_step_size(r, y[1])
        y_new = rk4_step(r, y, h, dm_case=dm_case)

        if y_new[0] <= 0 or y_new[1] <= 0 or np.any(np.isnan(y_new)) or np.any(np.isinf(y_new)):
            break

        r = r + h
        y = y_new

        r_list.append(r)
        rho_list.append(y[0])
        T_list.append(y[1])
        M_list.append(y[2])
        L_list.append(y[3])
        tau_list.append(y[4])

        derivs = stellar_derivatives(r, y, dm_case=dm_case)
        drhodr = derivs[0]
        rho = max(y[0], 1.0e-30)
        T = max(y[1], 1.0)
        kappa = opacity_total(rho, T)

        if abs(drhodr) > 1.0e-40:
            delta_tau = kappa * rho**2 / abs(drhodr)
        else:
            delta_tau = 1.0e99

        if delta_tau < 1.0e-3 and T < 2.0e4:
            converged_outer = True
            break

        if y[2] > 1000.0 * M_sun:
            break
        if r > 200.0 * R_sun:
            break

    history = {
        'r': np.array(r_list),
        'rho': np.array(rho_list),
        'T': np.array(T_list),
        'M': np.array(M_list),
        'L': np.array(L_list),
        'tau': np.array(tau_list),
        'outer_ok': converged_outer,
    }

    if len(history['r']) < 3:
        return None

    surface = interpolate_surface_from_tau(history)

    P_tot = []
    P_deg_list = []
    P_gas_list = []
    P_rad_list = []
    kappa_list = []
    k_es_list = []
    k_ff_list = []
    k_hm_list = []
    eps_pp_list = []
    eps_cno_list = []
    dLdr_list = []
    dLdr_pp_list = []
    dLdr_cno_list = []
    dLdr_dm_list = []
    rho_dm_list = []
    P_dm_list = []
    convective_flag = []

    for i in range(len(history['r'])):
        rr = history['r'][i]
        rho = history['rho'][i]
        T = history['T'][i]
        M = history['M'][i]
        L = history['L'][i]
        tau = history['tau'][i]

        P = pressure_total(rho, T)
        P_deg, P_gas, P_rad = pressure_parts(rho, T)
        k_es, k_ff, k_hm = opacity_parts(rho, T)
        kappa = opacity_total(rho, T)
        eps_pp_val = epsilon_pp(rho, T)
        eps_cno_val = epsilon_cno(rho, T)
        dLdr_pp = 4.0 * pi * rr**2 * rho * eps_pp_val
        dLdr_cno = 4.0 * pi * rr**2 * rho * eps_cno_val
        dLdr_dm_val = dLdr_DM(rr, dm_case=dm_case)
        dLdr_val = dLdr_pp + dLdr_cno + dLdr_dm_val

        deriv = stellar_derivatives(rr, np.array([rho, T, M, L, tau]), dm_case=dm_case)
        dTdr = deriv[1]
        dPdr = dP_drho(rho, T) * deriv[0] + dP_dT(rho, T) * dTdr
        if abs(dTdr) > 1.0e-300:
            dlogP_dlogT = (T / max(P, 1.0e-300)) * (dPdr / dTdr)
        else:
            dlogP_dlogT = np.inf
        is_convective = dlogP_dlogT <= gamma_ad / (gamma_ad - 1.0)

        P_tot.append(P)
        P_deg_list.append(P_deg)
        P_gas_list.append(P_gas)
        P_rad_list.append(P_rad)
        kappa_list.append(kappa)
        k_es_list.append(k_es)
        k_ff_list.append(k_ff)
        k_hm_list.append(k_hm)
        eps_pp_list.append(eps_pp_val)
        eps_cno_list.append(eps_cno_val)
        dLdr_list.append(dLdr_val)
        dLdr_pp_list.append(dLdr_pp)
        dLdr_cno_list.append(dLdr_cno)
        dLdr_dm_list.append(dLdr_dm_val)
        rho_dm_list.append(rho_DM_profile(rr, dm_case=dm_case))
        P_dm_list.append(power_density_DM(rr, dm_case=dm_case))
        convective_flag.append(is_convective)

    result = {
        'r': history['r'],
        'rho': history['rho'],
        'T': history['T'],
        'M': history['M'],
        'L': history['L'],
        'tau': history['tau'],
        'P': np.array(P_tot),
        'P_deg': np.array(P_deg_list),
        'P_gas': np.array(P_gas_list),
        'P_rad': np.array(P_rad_list),
        'kappa': np.array(kappa_list),
        'k_es': np.array(k_es_list),
        'k_ff': np.array(k_ff_list),
        'k_hm': np.array(k_hm_list),
        'eps_pp': np.array(eps_pp_list),
        'eps_cno': np.array(eps_cno_list),
        'dLdr': np.array(dLdr_list),
        'dLdr_pp': np.array(dLdr_pp_list),
        'dLdr_cno': np.array(dLdr_cno_list),
        'dLdr_dm': np.array(dLdr_dm_list),
        'rho_dm': np.array(rho_dm_list),
        'P_dm': np.array(P_dm_list),
        'convective': np.array(convective_flag, dtype=bool),
        'surface': surface,
        'outer_ok': history['outer_ok'],
        'dm_case': dm_case,
    }

    if result['surface'] is not None:
        result['surface']['Teff'] = effective_temperature_from_surface(result['surface'])

    return result

def luminosity_surface_error(rho_c, T_c, dm_case=None):
    """
    Return the mismatch function used in the shooting method.
    Root of this function corresponds to a star satisfying
    L(R*) = 4*pi*sigma*R*^2*T*^4
    """
    result = integrate_star(rho_c, T_c, dm_case=dm_case)
    if result is None or result.get('surface') is None:
        return None, None

    R_star = result['surface']['r']
    T_star = result['surface']['T']
    L_star = result['surface']['L']

    L_photo = 4.0 * pi * sigma_sb * R_star**2 * T_star**4

    denom = np.sqrt(max(L_star * L_photo, 1.0e-60))
    error = (L_star - L_photo) / denom

    return error, result

def solve_star_for_Tc(T_c, rho_guess=None, mass_guess=None, dm_case=None):
    """Solve for the central density rho_c at a chosen central temperature T_c."""

    rho_min = 3.0e2
    rho_max = 1.0e6

    def get_error_value(result):
        if result is None:
            return None

        if isinstance(result, (tuple, list)):
            if len(result) == 0:
                return None
            result = result[0]

        try:
            result = float(result)
        except (TypeError, ValueError):
            return None

        if not np.isfinite(result):
            return None

        return result

    def local_rho_grid(center):
        rho_low = max(rho_min, center / 10.0)
        rho_high = min(rho_max, center * 10.0)

        if rho_low >= rho_high:
            return np.array([center], dtype=float)

        return np.logspace(np.log10(rho_low), np.log10(rho_high), 25)

    def global_rho_grid():
        return np.logspace(np.log10(rho_min), np.log10(rho_max), 90)

    def candidate_score(model, rho_solution):
        if model is None or 'surface' not in model or model['surface'] is None:
            return np.inf

        s = model['surface']
        vals = [s['r'], s['M'], s['L'], get_surface_teff(s)]
        if not np.all(np.isfinite(vals)) or np.any(np.array(vals) <= 0.0):
            return np.inf

        score = 0.0

        if not model.get('outer_ok', False):
            score += 4.0

        Teff = get_surface_teff(s)
        if Teff > 2.0e5:
            score += 2.0
        elif Teff > 5.0e4:
            score += 0.5

        if mass_guess is not None and mass_guess > 0.0:
            score += 0.3 * abs(np.log(s['M'] / mass_guess))

        if rho_guess is not None and rho_guess > 0.0:
            score += 0.1 * abs(np.log(rho_solution / rho_guess))

        return score

    def find_all_sign_changes(rho_values):
        brackets = []
        prev_rho = None
        prev_err = None

        for rho in rho_values:
            raw_result = luminosity_surface_error(rho, T_c, dm_case=dm_case)
            err = get_error_value(raw_result)

            if err is None:
                continue

            if prev_err is not None and prev_err * err < 0:
                brackets.append((prev_rho, rho))

            prev_rho = rho
            prev_err = err

        return brackets

    def refine_bracket(a, b):
        fa = get_error_value(luminosity_surface_error(a, T_c, dm_case=dm_case))
        fb = get_error_value(luminosity_surface_error(b, T_c, dm_case=dm_case))

        if fa is None or fb is None:
            return None, None

        best_model = None
        best_rho = None
        best_abs_err = np.inf

        for _ in range(70):
            mid = np.sqrt(a * b)
            fm, model_mid = luminosity_surface_error(mid, T_c, dm_case=dm_case)
            fm = get_error_value(fm)

            if fm is None:
                return None, None

            if abs(fm) < best_abs_err and model_mid is not None:
                best_abs_err = abs(fm)
                best_model = model_mid
                best_rho = mid

            if abs(fm) < 1.0e-3 or abs(np.log10(b / a)) < 1.0e-4:
                return best_model, best_rho

            if fa * fm < 0:
                b = mid
                fb = fm
            else:
                a = mid
                fa = fm

        return best_model, best_rho

    rho_grids = []
    if rho_guess is not None:
        rho_grids.append(local_rho_grid(rho_guess))
    rho_grids.append(global_rho_grid())

    all_brackets = []
    seen = set()
    for grid in rho_grids:
        for bracket in find_all_sign_changes(grid):
            key = (round(np.log10(bracket[0]), 6), round(np.log10(bracket[1]), 6))
            if key not in seen:
                seen.add(key)
                all_brackets.append(bracket)

    candidates = []
    for a, b in all_brackets:
        model, rho_solution = refine_bracket(a, b)
        if model is None or rho_solution is None:
            continue
        score = candidate_score(model, rho_solution)
        if np.isfinite(score):
            candidates.append((score, model, rho_solution))

    if not candidates:
        return None, None

    candidates.sort(key=lambda item: item[0])
    return candidates[0][1], candidates[0][2]

def build_main_sequence(Tc_values, dm_case=None, sequence_name='Standard'):
    """
    Build the main-sequence models by starting at high Tc and stepping
    downward, using both previous rho_c and previous mass to stay on
    one smooth physical branch.
    """
    models = []

    Tc_values = np.array(sorted(Tc_values, reverse=True), dtype=float)

    previous_rho_c = None
    previous_mass = None

    if dm_case is None:
        print(f"Building sequence: {sequence_name}")
    else:
        print(f"Building sequence: {dm_case['label']}")

    for Tc in Tc_values:
        print(f"Solving model for Tc = {Tc:.3e} K")

        model, rho_solution = solve_star_for_Tc(
            Tc,
            rho_guess=previous_rho_c,
            mass_guess=previous_mass,
            dm_case=dm_case
        )

        if model is not None:
            models.append(model)
            previous_rho_c = rho_solution
            previous_mass = model['surface']['M']

            s = model['surface']
            print(
                "  found: "
                f"M={s['M']/M_sun:.3f} Msun, "
                f"R={s['r']/R_sun:.3f} Rsun, "
                f"L={s['L']/L_sun:.3e} Lsun, "
                f"Teff={get_surface_teff(s):.0f} K, "
                f"rho_c={rho_solution:.3e}"
            )
        else:
            print("  no solution found")

    return models

def select_generated_high_mass_model(models, min_mass_msun=2.0):
    """
    Choose a generated stellar model strictly above the required mass threshold.

    Returns the model whose mass is the smallest value still above the threshold.

    Returns None if no qualifying model exists.
    """
    candidates = [m for m in models if m['surface']['M'] / M_sun > min_mass_msun]

    if len(candidates) > 0:
        return min(candidates, key=lambda m: (m['surface']['M'] / M_sun) - min_mass_msun)

    return None


def select_generated_low_mass_model(models, max_mass_msun=0.75):
    """
    Choose a generated stellar model at or below the low-mass threshold.

    Returns the qualifying model whose mass is closest to the threshold from below.
    Returns None if no qualifying model exists.
    """
    candidates = [m for m in models if m['surface']['M'] / M_sun <= max_mass_msun]

    if len(candidates) > 0:
        return max(candidates, key=lambda m: m['surface']['M'] / M_sun)

    return None


def describe_model(model, header):
    """Print a short summary of one model for inspection."""
    print(f'\n{header}')
    print(f"  M = {model['surface']['M']/M_sun:.3f} Msun")
    print(f"  R = {model['surface']['r']/R_sun:.3f} Rsun")
    print(f"  L = {model['surface']['L']/L_sun:.3e} Lsun")
    print(f"  Teff = {get_surface_teff(model['surface']):.0f} K")


# Plotting helpers

def annotate_example_points(ax, x_low, y_low, x_high, y_high):
    """Add separate labels for the low-mass and high-mass stars."""
    ax.annotate('Low-mass star', xy=(x_low, y_low), xytext=(8, 8),
                textcoords='offset points')
    ax.annotate('High-mass star', xy=(x_high, y_high), xytext=(8, -12),
                textcoords='offset points')


def plot_main_sequence(models, example_stars=None):
    masses = np.array([m['surface']['M'] / M_sun for m in models])
    radii = np.array([m['surface']['r'] / R_sun for m in models])
    luminosities = np.array([m['surface']['L'] / L_sun for m in models])
    teff = np.array([effective_temperature_from_surface(m['surface']) for m in models])

    # HR diagram
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(teff, luminosities,marker= '*',s=30,color='mediumvioletred', label='Solved sequence')

    if example_stars is not None:
        low_teff = example_stars['low']['surface']['Teff']
        high_teff = example_stars['high']['surface']['Teff']
        low_lum = example_stars['low']['surface']['L'] / L_sun
        high_lum = example_stars['high']['surface']['L'] / L_sun
        ax.scatter(low_teff, low_lum, marker='d', color = 'cornflowerblue', label='Low-mass star', zorder=2)
        ax.scatter(high_teff, high_lum, marker='o', color = 'seagreen', label='High-mass star', zorder=2)
        annotate_example_points(ax, low_teff, low_lum, high_teff, high_lum)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.invert_xaxis()
    ax.set_xlabel('Effective Temperature Teff (K)')
    ax.set_ylabel('Luminosity (L/Lsun)')
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xscale('log')            
        
    spectral_teff = [30000, 20000, 10000, 7000, 5500, 4500, 3200]
    spectral_labels = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
    
    ax_top.set_xticks(spectral_teff)
    ax_top.tick_params(direction='in')
    ax_top.set_xticklabels(spectral_labels)
    ax_top.set_xlabel('Spectral Type')
    ax.set_title('Model HR Diagram')
    ax.grid(True, alpha=0.3)
    if example_stars is not None:
        ax.legend()
    plt.show()

    # L-M relation
    fig, ax = plt.subplots(figsize=(7, 5))
    idx = np.argsort(masses)
    ax.plot(masses[idx], luminosities[idx], '*-',color='mediumvioletred', label='Solved sequence')
    if example_stars is not None:
        low_mass = example_stars['low']['surface']['M'] / M_sun
        high_mass = example_stars['high']['surface']['M'] / M_sun
        low_lum = example_stars['low']['surface']['L'] / L_sun
        high_lum = example_stars['high']['surface']['L'] / L_sun
        ax.scatter(low_mass, low_lum, marker='d', color = 'cornflowerblue', label='Low-mass star', zorder=2)
        ax.scatter(high_mass, high_lum, marker='o', color = 'seagreen', label='High-mass star', zorder=2)
        annotate_example_points(ax, low_mass, low_lum, high_mass, high_lum)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Mass (M/Msun)')
    ax.set_ylabel('Luminosity (L/Lsun)')
    ax.set_title('Luminosity-Mass Relation')
    ax.grid(True, alpha=0.3)
    if example_stars is not None:
        ax.legend()
    plt.show()

    # R-M relation
    fig, ax = plt.subplots(figsize=(7, 5))
    idx = np.argsort(masses)
    ax.plot(masses[idx], radii[idx], 'o-',color='mediumvioletred', label='Solved sequence')
    if example_stars is not None:
        low_mass = example_stars['low']['surface']['M'] / M_sun
        high_mass = example_stars['high']['surface']['M'] / M_sun
        low_radius = example_stars['low']['surface']['r'] / R_sun
        high_radius = example_stars['high']['surface']['r'] / R_sun
        ax.scatter(low_mass, low_radius, marker='d', color = 'cornflowerblue', label='Low-mass star', zorder=2)
        ax.scatter(high_mass, high_radius, marker='o', color = 'seagreen', label='High-mass star', zorder=2)
        annotate_example_points(ax, low_mass, low_radius, high_mass, high_radius)
    ax.minorticks_on()
    ax.tick_params(which='major', length=7, width=1, direction='in')
    ax.tick_params(which='minor', length=7, width=1, direction='in')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Mass (M/Msun)')
    ax.set_ylabel('Radius (R/Rsun)')
    ax.set_title('Radius-Mass Relation')
    ax.grid(True, alpha=0.3)
    if example_stars is not None:
        ax.legend()
    plt.show()



def clean_convective_mask(x, convective, min_zone_width=0.03, min_gap_width=0.015):
    """
    Clean a noisy convective mask for plotting by merging tiny radiative gaps and
    removing tiny isolated convective slivers. This restores visually meaningful
    convection zones when the stability criterion fluctuates near the boundary.
    Widths are in units of x = r/R_*.
    """
    x = np.asarray(x, dtype=float)
    mask = np.asarray(convective, dtype=bool).copy()

    if mask.size < 3:
        return mask

    # Make sure x is monotonic enough for width estimates
    order = np.argsort(x)
    x = x[order]
    mask = mask[order]

    def run_lengths(arr):
        runs = []
        start = 0
        n = len(arr)
        while start < n:
            value = arr[start]
            end = start
            while end + 1 < n and arr[end + 1] == value:
                end += 1
            runs.append((start, end, value))
            start = end + 1
        return runs

    # First, fill very small radiative gaps inside otherwise convective regions.
    changed = True
    while changed:
        changed = False
        runs = run_lengths(mask)
        for i, (start, end, value) in enumerate(runs):
            if value:
                continue
            left_true = i > 0 and runs[i - 1][2]
            right_true = i < len(runs) - 1 and runs[i + 1][2]
            if left_true and right_true:
                width = x[end] - x[start]
                if width <= min_gap_width:
                    mask[start:end + 1] = True
                    changed = True
                    break

    # Then remove tiny isolated convective slivers.
    changed = True
    while changed:
        changed = False
        runs = run_lengths(mask)
        for i, (start, end, value) in enumerate(runs):
            if not value:
                continue
            width = x[end] - x[start]
            if width <= min_zone_width:
                mask[start:end + 1] = False
                changed = True
                break

    # Put mask back into original ordering.
    cleaned = np.zeros_like(mask, dtype=bool)
    cleaned[np.argsort(order)] = mask
    return cleaned


def shade_convection(ax, x, convective):
    """Shade convective regions."""
    x = np.asarray(x, dtype=float)
    convective = clean_convective_mask(x, convective)

    in_zone = False
    start = None
    for i in range(len(x)):
        if convective[i] and not in_zone:
            start = x[i]
            in_zone = True
        if in_zone and (i == len(x) - 1 or not convective[i + 1]):
            end = x[i]
            if end > start:
                ax.axvspan(start, end, color='gray', alpha=0.2)
            in_zone = False


def plot_star_profiles(model, title_prefix='Star'):
    surface = model['surface']
    R_star = surface['r']
    x = model['r'] / R_star

    rho_norm = model['rho'] / model['rho'][0]
    T_norm = model['T'] / model['T'][0]
    M_norm = model['M'] / model['surface']['M']
    L_norm = model['L'] / model['surface']['L']
    P_norm = model['P'] / model['P'][0]

    fig, axes = plt.subplots(3, 2, figsize=(13.5, 12), sharex=True)
    fig.subplots_adjust(top=0.88, wspace=0.42, hspace=0.28)
    axes = axes.flatten()

    ax = axes[0]
    ax.plot(x, rho_norm, label=r'$\rho / \rho_c$', color='navy', linestyle=(5, (10, 3)))
    ax.plot(x, T_norm, label=r'$T/T_c$', color='mediumvioletred')
    ax.plot(x, M_norm, label=r'$M/M_*$', color='seagreen', linestyle='-.')
    ax.plot(x, L_norm, label=r'$L/L_*$', color='mediumpurple', linestyle='--')
    shade_convection(ax, x, model['convective'])
    ax.set_xlabel(r'$r / R_*$')
    ax.set_ylabel(r'$T/T_c, \rho/\rho_c, M/M_*, L/L_*$')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(x, P_norm, label=r'$P_{total} / P_c$')
    ax.plot(x, model['P_deg'] / model['P'][0], '--', label=r'$P_{deg} / P_c$')
    ax.plot(x, model['P_gas'] / model['P'][0], '-.', label=r'$P_{gas} / P_c$')
    ax.plot(x, model['P_rad'] / model['P'][0], linestyle=(5, (10, 3)), label=r'$P_{rad} / P_c$')
    shade_convection(ax, x, model['convective'])
    ax.set_xlabel(r'$r / R_*$')
    ax.set_ylabel(r'$P_{deg} / P_c, P_{gas} / P_c, P_{rad} / P_c$ ')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(x, model['kappa'], label=r'$\kappa_{total}$')
    ax.plot(x, model['k_es'], '-.', label=r'$\kappa_{es}$')
    ax.plot(x, model['k_ff'], '--', label=r'$\kappa_{ff}$')
    ax.plot(x, model['k_hm'], linestyle=(5, (10, 3)), label=r'$\kappa_{H-}$')
    shade_convection(ax, x, model['convective'])
    ax.set_yscale('log')
    ax.set_xlabel('r / R*')
    ax.set_ylabel('Opacity (m^2/kg)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[3]
    norm = max(model['surface']['L'] / max(surface['r'], 1.0), 1.0e-99)
    dLdr_total = model.get('dLdr', np.zeros_like(model['r']))
    dLdr_pp = model.get('dLdr_pp')
    if dLdr_pp is None:
        if 'eps_pp' in model:
            dLdr_pp = 4.0 * pi * model['r']**2 * np.maximum(model['rho'], 0.0) * np.maximum(model['eps_pp'], 0.0)
        else:
            dLdr_pp = np.zeros_like(model['r'])
    dLdr_cno = model.get('dLdr_cno')
    if dLdr_cno is None:
        if 'eps_cno' in model:
            dLdr_cno = 4.0 * pi * model['r']**2 * np.maximum(model['rho'], 0.0) * np.maximum(model['eps_cno'], 0.0)
        else:
            dLdr_cno = np.zeros_like(model['r'])
    dLdr_dm = model.get('dLdr_dm', np.zeros_like(model['r']))

    ax.plot(x, dLdr_total / norm, label='dL/dr total')
    ax.plot(x, dLdr_pp / norm, '--', label='dL/dr PP')
    ax.plot(x, dLdr_cno / norm, '-.', label='dL/dr CNO')
    if np.any(np.asarray(dLdr_dm) > 0.0):
        ax.plot(x, dLdr_dm / norm, ':', label='dL/dr DM')
    shade_convection(ax, x, model['convective'])
    ax.set_xlabel(r'$r / R_*$')
    ax.set_ylabel('dL/dr (scaled)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[4]
    ax.plot(x, model['rho'], label='rho')
    shade_convection(ax, x, model['convective'])
    ax.set_yscale('log')
    ax.set_xlabel(r'$r / R_*$')
    ax.set_ylabel('Density (kg/m^3)')
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(x, model['T'], 'r--', label='T')
    ax2.set_ylabel('Temperature (K)', labelpad=12)
    ax2.set_yscale('log')

    ax = axes[5]
    ax.plot(x, M_norm, label='M/M_*')
    ax.plot(x, L_norm, '--', label='L/L_*')
    shade_convection(ax, x, model['convective'])
    ax.set_xlabel(r'$r / R_*$')
    ax.set_ylabel('Fraction enclosed')
    ax.legend()
    ax.grid(True, alpha=0.3)

    dm_label = ''
    if model.get('dm_case') is not None:
        dm_label = '\n' + model['dm_case']['label']

    fig.suptitle(
        f"{title_prefix}\n"
        f"M={surface['M']/M_sun:.3f} Msun, R={surface['r']/R_sun:.3f} Rsun, "
        f"L={surface['L']/L_sun:.3e} Lsun, Teff={get_surface_teff(surface):.0f} K"
        + dm_label
    )

    for ax in axes:
        ax.minorticks_on()
        ax.tick_params(which='major', length=7, width=1, direction='in')
        ax.tick_params(which='minor', length=3, width=1, direction='in')

    for ax in axes.flat:
        for spine in ax.spines.values():
            spine.set_linewidth(1.2)
        ax.tick_params(direction='in', which='both', top=True, right=True)

    plt.show()


def comparison_plot_style(index, label):
    """Return a distinct style dictionary for comparison plots."""
    if label == 'Standard':
        return dict(color='black', marker='o', linestyle='-', linewidth=1.6,
                    markersize=4, markerfacecolor='black', markeredgewidth=0.8)

    dm_styles = [
        dict(color='mediumvioletred', marker='o', linestyle='-', linewidth=1.6,
             markersize=4, markerfacecolor='mediumvioletred', markeredgewidth=0.8),
        dict(color='seagreen', marker='s', linestyle='--', linewidth=1.6,
             markersize=4.5, markerfacecolor='seagreen', markeredgewidth=0.8),
        dict(color='cornflowerblue', marker='D', linestyle='-.', linewidth=1.6,
             markersize=4.2, markerfacecolor='cornflowerblue', markeredgewidth=0.8),
        dict(color='darkorange', marker='^', linestyle=':', linewidth=1.8,
             markersize=4.8, markerfacecolor='darkorange', markeredgewidth=0.8),
        dict(color='teal', marker='P', linestyle=(0, (5, 2, 1, 2)), linewidth=1.6,
             markersize=4.6, markerfacecolor='teal', markeredgewidth=0.8),
    ]
    return dm_styles[(index - 1) % len(dm_styles)]


def plot_main_sequence_comparison(sequence_entries):
    """Compare standard and DM-heated sequences on the HR, L-M, and R-M plots."""
    if not sequence_entries:
        return

    fig1, ax1 = plt.subplots(figsize=(8, 6))
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    fig3, ax3 = plt.subplots(figsize=(8, 6))

    for idx_entry, entry in enumerate(sequence_entries):
        models = entry['models']
        if not models:
            continue

        masses = np.array([m['surface']['M'] / M_sun for m in models])
        radii = np.array([m['surface']['r'] / R_sun for m in models])
        luminosities = np.array([m['surface']['L'] / L_sun for m in models])
        teff = np.array([get_surface_teff(m['surface']) for m in models])

        idx_mass = np.argsort(masses)
        style = comparison_plot_style(idx_entry, entry['label'])

        ax1.plot(teff, luminosities, label=entry['label'], **style)
        ax2.plot(masses[idx_mass], luminosities[idx_mass], label=entry['label'], **style)
        ax3.plot(masses[idx_mass], radii[idx_mass], label=entry['label'], **style)

    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.invert_xaxis()
    ax1.set_xlabel('Effective Temperature Teff (K)')
    ax1.set_ylabel('Luminosity (L/Lsun)')
    ax1.set_title('DM Heating comparison: HR diagram')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=9)

    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel('Mass (M/Msun)')
    ax2.set_ylabel('Luminosity (L/Lsun)')
    ax2.set_title('DM Heating comparison: Luminosity-Mass relation')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=9)

    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_xlabel('Mass (M/Msun)')
    ax3.set_ylabel('Radius (R/Rsun)')
    ax3.set_title('DM Heating comparison: Radius-Mass relation')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=9)

    plt.show()

def main():
    Tc_values = np.array([
        6.00000e6,
        7.00000e6,
        7.40800e6,
        8.23544e6,
        9.14600e6,
        1.05000e7,
        1.12900e7,
        1.25000e7,
        1.39400e7,
        1.55000e7,
        1.72100e7,
        1.90000e7,
        2.00000e7,
        2.12500e7,
        2.35000e7,
        2.62400e7,
        3.00000e7,
        3.24000e7,
        4.00000e7
    ], dtype=float)

    example_stars = load_example_stars_once(DATA_FOLDER)
    print(f'Loaded example-star files from: {os.path.abspath(DATA_FOLDER)}')

    models = build_main_sequence(Tc_values, dm_case=None, sequence_name='Standard')

    if len(models) == 0:
        print('No stellar models were found.')
        return

    models.sort(key=lambda m: m['surface']['M'])

    generated_high_star = select_generated_high_mass_model(models, min_mass_msun=2.0)
    if generated_high_star is None:
        print('No generated standard-sequence model above 2 Msun was found.')
        return

    stars_to_highlight = {
        'low': example_stars['low'],
        'high': generated_high_star,
    }
    plot_main_sequence(models, example_stars=stars_to_highlight)

    low_star = example_stars['low']
    print('\nLow-mass star (provided reference file):')
    print(f"  M = {low_star['surface']['M']/M_sun:.3f} Msun")
    print(f"  R = {low_star['surface']['r']/R_sun:.3f} Rsun")
    print(f"  L = {low_star['surface']['L']/L_sun:.3e} Lsun")
    print(f"  Teff = {get_surface_teff(low_star['surface']):.0f} K")
    plot_star_profiles(low_star, title_prefix='Low-Mass Reference Star')

    high_star = generated_high_star
    print('\nHigh-mass star (generated sequence model above 2 Msun):')
    print(f"  M = {high_star['surface']['M']/M_sun:.3f} Msun")
    print(f"  R = {high_star['surface']['r']/R_sun:.3f} Rsun")
    print(f"  L = {high_star['surface']['L']/L_sun:.3e} Lsun")
    print(f"  Teff = {get_surface_teff(high_star['surface']):.0f} K")
    plot_star_profiles(high_star, title_prefix='Generated High-Mass Model (> 2 Msun)')

    if RUN_SECTION_57_COMPARISON:
        sequence_entries = [{'label': 'Standard', 'models': models}]
        dm_case_entries = []

        for dm_case in representative_section57_cases():
            dm_models = build_main_sequence(Tc_values, dm_case=dm_case, sequence_name=dm_case['label'])
            dm_models.sort(key=lambda m: m['surface']['M'])
            sequence_entries.append({'label': dm_case['label'], 'models': dm_models})
            dm_case_entries.append({'case': dm_case, 'models': dm_models})

        plot_main_sequence_comparison(sequence_entries)

        for entry in dm_case_entries:
            dm_case = entry['case']
            dm_models = entry['models']

            if len(dm_models) == 0:
                print(f"\nNo models found for {dm_case['label']}; skipping DM inspection plots.")
                continue

            low_dm_star = select_generated_low_mass_model(dm_models, max_mass_msun=0.75)
            high_dm_star = select_generated_high_mass_model(dm_models, min_mass_msun=2.0)

            if low_dm_star is not None:
                describe_model(low_dm_star, f"Low-mass DM-heated model ({dm_case['label']}):")
                plot_star_profiles(low_dm_star, title_prefix=f"Low-Mass DM-Heated Model\n{dm_case['label']}")
            else:
                print(f"\nNo DM-heated model below or at 0.75 Msun was found for {dm_case['label']}; skipping low-mass inspection.")

            if high_dm_star is not None:
                describe_model(high_dm_star, f"High-mass DM-heated model ({dm_case['label']}):")
                plot_star_profiles(high_dm_star, title_prefix=f"High-Mass DM-Heated Model\n{dm_case['label']}")
            else:
                print(f"\nNo DM-heated model above 2 Msun was found for {dm_case['label']}; skipping high-mass inspection.")


if __name__ == '__main__':
    main()


Loaded example-star files from: /mnt/zfs/jupyter-p03/home/b6carter/Standard MS System repaired
Building sequence: Standard
Solving model for Tc = 4.000e+07 K
  found: M=183.950 Msun, R=9.407 Rsun, L=2.312e+07 Lsun, Teff=130502 K, rho_c=3.286e+02
Solving model for Tc = 3.240e+07 K
  found: M=29.295 Msun, R=6.088 Rsun, L=9.481e+04 Lsun, Teff=41050 K, rho_c=1.511e+03
Solving model for Tc = 3.000e+07 K
  found: M=15.144 Msun, R=4.239 Rsun, L=2.006e+04 Lsun, Teff=33363 K, rho_c=3.367e+03
Solving model for Tc = 2.624e+07 K
  found: M=6.346 Msun, R=2.055 Rsun, L=1.808e+03 Lsun, Teff=26252 K, rho_c=1.121e+04
Solving model for Tc = 2.350e+07 K
  found: M=3.431 Msun, R=1.260 Rsun, L=2.769e+02 Lsun, Teff=20975 K, rho_c=2.820e+04
Solving model for Tc = 2.125e+07 K
  found: M=2.150 Msun, R=0.998 Rsun, L=5.562e+01 Lsun, Teff=15780 K, rho_c=6.037e+04
Solving model for Tc = 2.000e+07 K
  found: M=1.712 Msun, R=0.734 Rsun, L=2.376e+01 Lsun, Teff=14874 K, rho_c=8.682e+04
Solving model for Tc = 1.900e+07